# TTS Backend Comparison — TTFA & Audio Quality

So sánh 3 config VieNeu-TTS-0.3B trong điều kiện streaming thực tế:
- **Config A**: VieNeuTTS-0.3B + GGUF INT4 (llama-cpp) — production config hiện tại
- **Config B**: VieNeuTTS-0.3B + PyTorch BF16 — full precision, cùng class
- **Config C**: FastVieNeuTTS-0.3B + LMDeploy BF16 — GPU-optimized, token streaming thật

Cả 3 config dùng cùng model 0.3B để so sánh công bằng (chỉ khác inference backend).  
LLM (Qwen3-4B) stream text thật → gateway chunker (min 40 chars) → TTS backend.

**Metrics:**
- TTFA (Time-To-First-Audio): ms từ khi query gửi đến chunk audio đầu tiên
- Total latency: ms đến khi audio hoàn chỉnh
- Audio files: lưu để CMOS evaluation

**VRAM estimate (A100 40GB):**
- Qwen3-4B (bitsandbytes int4): ~10 GB (gpu-memory-utilization=0.25)
- Config A GGUF INT4: ~1 GB → tổng ~11 GB ✅
- Config B PyTorch BF16: ~2 GB → tổng ~12 GB ✅
- Config C LMDeploy BF16: ~3 GB + overhead → tổng ~14 GB ✅
- → Chạy từng config một, giải phóng GPU trước khi load config tiếp theo

## 1. Install dependencies

In [ ]:
# Phần 1: vLLM để host Qwen3-4B
!pip install -q vllm bitsandbytes

# Phần 2: VieNeu-TTS (CPU + GGUF backend)
!pip install -q 'vieneu[cpu]'
# llama-cpp-python với CUDA support
!CMAKE_ARGS='-DGGML_CUDA=on' pip install -q llama-cpp-python --upgrade --force-reinstall --no-cache-dir

# Phần 3: LMDeploy (cho Config C)
!pip install -q lmdeploy

# Phần 4: Audio + misc
!pip install -q soundfile openai httpx asyncio-compat

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.1f} GB")

## 2. Start vLLM server (Qwen3-4B)

In [ ]:
import subprocess
import time
import requests

vllm_proc = subprocess.Popen(
    [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", "Qwen/Qwen3-4B-Instruct-2507",
        "--dtype", "half",
        "--quantization", "bitsandbytes",
        "--load-format", "bitsandbytes",
        "--max-model-len", "2048",
        "--gpu-memory-utilization", "0.25",  # ~10GB → còn ~30GB cho TTS
        "--max-num-seqs", "4",
        "--port", "8000",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

print("Waiting for vLLM to start...")
for _ in range(60):
    time.sleep(5)
    try:
        r = requests.get("http://localhost:8000/health", timeout=3)
        if r.status_code == 200:
            print("vLLM ready!")
            break
    except Exception:
        pass
else:
    # Print last lines of log to debug
    import os, fcntl
    flags = fcntl.fcntl(vllm_proc.stdout, fcntl.F_GETFL)
    fcntl.fcntl(vllm_proc.stdout, fcntl.F_SETFL, flags | os.O_NONBLOCK)
    try:
        print(vllm_proc.stdout.read(4096).decode())
    except Exception:
        pass
    raise RuntimeError("vLLM did not start in time")

## 3. Setup: System prompt, queries, helpers

In [ ]:
import asyncio
import re
import time
import threading
import numpy as np
import soundfile as sf
from pathlib import Path
from openai import AsyncOpenAI

SAMPLE_RATE = 24_000
OUTPUT_DIR = Path("tts_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

llm_client = AsyncOpenAI(
    base_url="http://localhost:8000/v1",
    api_key="dummy",
)

SYSTEM_PROMPT = """Bạn là chuyên gia tư vấn dinh dưỡng qua giọng nói. Tuân thủ:
1. Trả lời dựa trên kiến thức dinh dưỡng. Không trích dẫn tên nguồn hay URL.
2. Bắt đầu bằng "Chào bạn,", tư vấn như chuyên gia dinh dưỡng.
3. Ngắn gọn, dễ nghe: tối đa 100 từ, dùng câu ngắn, không dùng bullet points.
4. Kết thúc bằng "Để được tư vấn chính xác, bạn nên gặp bác sĩ dinh dưỡng."
"""

# ── Load queries từ file — đổi đường dẫn nếu cần ────────────────────────────
QUERIES_FILE = "/path/to/tts_eval_queries.txt"  # ← đổi đường dẫn ở đây

with open(QUERIES_FILE, encoding="utf-8") as f:
    TEST_QUERIES = [line.strip() for line in f if line.strip()]

print(f"Loaded {len(TEST_QUERIES)} queries")
print("First 3:", TEST_QUERIES[:3])

# ── Gateway chunker (copy từ brain/core/chunker.py) ──────────────────────────
PUNCTUATION = re.compile(r"[.!?;:,]")
MIN_CHUNK_SIZE = 40

async def chunk_llm_stream(text_stream, min_size=MIN_CHUNK_SIZE):
    buffer = ""
    async for token in text_stream:
        buffer += re.sub(r"\n+", " ", token)
        while len(buffer) >= min_size:
            match = None
            for m in PUNCTUATION.finditer(buffer):
                if m.start() >= min_size // 2:
                    match = m
            if match:
                cut = match.end()
                yield buffer[:cut].rstrip()
                buffer = buffer[cut:]
            elif len(buffer) > min_size * 2:
                sp = buffer.rfind(" ", min_size // 2, min_size * 2)
                if sp > 0:
                    yield buffer[:sp].rstrip()
                    buffer = buffer[sp:]
                else:
                    break
            else:
                break
    if buffer.strip():
        yield buffer.rstrip()

def pcm_float_to_int16_bytes(audio: np.ndarray) -> bytes:
    return (audio * 32767).clip(-32768, 32767).astype(np.int16).tobytes()

def save_wav(pcm_chunks: list[bytes], path: Path):
    raw = b"".join(pcm_chunks)
    audio = np.frombuffer(raw, dtype=np.int16).astype(np.float32) / 32768.0
    sf.write(str(path), audio, SAMPLE_RATE)

print("Setup done.")

## 3b. Pre-generate LLM responses (chạy 1 lần, dùng lại cho 3 config)

`temperature=0` → output deterministic → cả 3 TTS config nhận đúng cùng text.

In [ ]:
import json

RESPONSES_FILE = "llm_responses.json"

async def generate_one(query: str) -> str:
    resp = await llm_client.chat.completions.create(
        model="Qwen/Qwen3-4B-Instruct-2507",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": query},
        ],
        stream=False,
        max_tokens=300,
        temperature=0,  # deterministic
    )
    return resp.choices[0].message.content

# Nếu đã có file thì load lại, không generate lại
if Path(RESPONSES_FILE).exists():
    with open(RESPONSES_FILE, encoding="utf-8") as f:
        llm_responses = json.load(f)
    print(f"Loaded {len(llm_responses)} cached responses from {RESPONSES_FILE}")
else:
    print(f"Generating {len(TEST_QUERIES)} responses (temperature=0)...")
    llm_responses = {}
    for i, query in enumerate(TEST_QUERIES):
        llm_responses[query] = await generate_one(query)
        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(TEST_QUERIES)} done")

    with open(RESPONSES_FILE, "w", encoding="utf-8") as f:
        json.dump(llm_responses, f, ensure_ascii=False, indent=2)
    print(f"Saved to {RESPONSES_FILE}")

# Kiểm tra độ dài response
lengths = [len(v) for v in llm_responses.values()]
print(f"\nResponse length — min={min(lengths)} max={max(lengths)} avg={sum(lengths)//len(lengths)} chars")
print(f"\nSample (query 1):\n  Q: {TEST_QUERIES[0]}\n  A: {llm_responses[TEST_QUERIES[0]][:120]}...")

## 4. Benchmark runner

In [ ]:
async def replay_as_stream(text: str, token_delay_ms: float = 20):
    """Replay pre-generated text như LLM đang stream — yield từng từ với delay."""
    for word in text.split():
        yield word + " "
        await asyncio.sleep(token_delay_ms / 1000)

async def run_benchmark(tts_instance, config_name: str, queries: list[str],
                        responses: dict, token_delay_ms: float = 20):
    """
    Với mỗi query:
      1. Replay pre-generated LLM response qua fake stream
      2. chunk_llm_stream buffer → text chunks (giống gateway)
      3. Mỗi text chunk → tts.infer_stream() trong thread
      4. Đo TTFA = time từ replay_start đến audio frame đầu tiên
      5. Lưu audio WAV
    """
    results = []
    voice = tts_instance.get_preset_voice()
    loop = asyncio.get_running_loop()

    for q_idx, query in enumerate(queries):
        if query not in responses:
            print(f"  [{config_name}] Query {q_idx+1}: no response cached, skip")
            continue

        full_text = responses[query]
        print(f"  [{config_name}] Q{q_idx+1:03d}/{len(queries)}: {query[:50]}...")

        pcm_chunks_all = []
        ttfa_ms = None
        start = time.perf_counter()

        async def text_chunks_gen():
            stream = replay_as_stream(full_text, token_delay_ms)
            async for tc in chunk_llm_stream(stream):
                yield tc

        async for text_chunk in text_chunks_gen():
            q: asyncio.Queue = asyncio.Queue()

            def _tts_worker(tc=text_chunk):
                try:
                    for audio_frame in tts_instance.infer_stream(tc, voice=voice):
                        loop.call_soon_threadsafe(q.put_nowait, audio_frame)
                except Exception as e:
                    loop.call_soon_threadsafe(q.put_nowait, e)
                finally:
                    loop.call_soon_threadsafe(q.put_nowait, None)

            threading.Thread(target=_tts_worker, daemon=True).start()

            while True:
                frame = await q.get()
                if frame is None:
                    break
                if isinstance(frame, Exception):
                    print(f"    TTS error: {frame}")
                    break
                if ttfa_ms is None:
                    ttfa_ms = (time.perf_counter() - start) * 1000
                pcm_chunks_all.append(pcm_float_to_int16_bytes(frame))

        total_ms = (time.perf_counter() - start) * 1000

        wav_path = OUTPUT_DIR / f"{config_name}_q{q_idx+1:03d}.wav"
        if pcm_chunks_all:
            save_wav(pcm_chunks_all, wav_path)

        result = {
            "config": config_name,
            "query_idx": q_idx + 1,
            "query": query,
            "response_chars": len(full_text),
            "ttfa_ms": round(ttfa_ms, 1) if ttfa_ms else None,
            "total_ms": round(total_ms, 1),
            "wav_path": str(wav_path),
        }
        results.append(result)
        print(f"    TTFA={result['ttfa_ms']}ms  Total={result['total_ms']}ms  resp={len(full_text)}chars")

    return results

def print_summary(name: str, results: list):
    import statistics
    ttfas = sorted([r["ttfa_ms"] for r in results if r["ttfa_ms"]])
    totals = sorted([r["total_ms"] for r in results])
    def p95(lst): return lst[min(int(len(lst)*0.95), len(lst)-1)]
    print(f"{name:<20}  TTFA median={statistics.median(ttfas):.0f}ms p95={p95(ttfas):.0f}ms  "
          f"Total median={statistics.median(totals):.0f}ms p95={p95(totals):.0f}ms")

print("Benchmark runner ready.")

## 5. Config A — VieNeuTTS GGUF INT4 (production)

In [ ]:
from vieneu.core import VieNeuTTS

print("Loading Config A: VieNeuTTS GGUF INT4...")
tts_a = VieNeuTTS(
    backbone_repo="pnnbao-ump/VieNeu-TTS-0.3B-q4-gguf",
    backbone_device="gpu",
    codec_repo="neuphonic/distill-neucodec",
    codec_device="cuda",
)

# Warmup
print("Warming up...")
_ = list(tts_a.infer_stream("Xin chào.", voice=tts_a.get_preset_voice()))
print("Config A ready.")

In [ ]:
results_a = await run_benchmark(tts_a, "A_GGUF_INT4", TEST_QUERIES, llm_responses)
print_summary("A_GGUF_INT4", results_a)

In [ ]:
# Giải phóng VRAM trước khi load config tiếp
tts_a.close()
del tts_a
import gc; gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.1f} GB")

## 6. Config B — VieNeuTTS-0.3B PyTorch BF16

In [ ]:
print("Loading Config B: VieNeuTTS-0.3B PyTorch BF16...")
tts_b = VieNeuTTS(
    backbone_repo="pnnbao-ump/VieNeu-TTS-0.3B",  # 0.3B full precision BF16
    backbone_device="cuda",
    codec_repo="neuphonic/distill-neucodec",
    codec_device="cuda",
)

print("Warming up...")
_ = list(tts_b.infer_stream("Xin chào.", voice=tts_b.get_preset_voice()))
print("Config B ready.")

In [ ]:
results_b = await run_benchmark(tts_b, "B_PyTorch_BF16", TEST_QUERIES, llm_responses)
print_summary("B_PyTorch_BF16", results_b)

In [ ]:
tts_b.close()
del tts_b
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.1f} GB")

## 7. Config C — FastVieNeuTTS-0.3B LMDeploy BF16

In [ ]:
from vieneu.core import FastVieNeuTTS

print("Loading Config C: FastVieNeuTTS-0.3B LMDeploy BF16...")
tts_c = FastVieNeuTTS(
    backbone_repo="pnnbao-ump/VieNeu-TTS-0.3B",  # 0.3B full precision BF16
    backbone_device="cuda",
    codec_repo="neuphonic/distill-neucodec",
    codec_device="cuda",
    memory_util=0.15,   # ~6GB VRAM cho TTS backbone (0.3B nhỏ hơn)
    max_batch_size=4,
    enable_triton=True,
)
# FastVieNeuTTS tự warmup trong __init__
print("Config C ready.")

In [ ]:
results_c = await run_benchmark(tts_c, "C_LMDeploy_BF16", TEST_QUERIES, llm_responses)
print_summary("C_LMDeploy_BF16", results_c)

In [ ]:
del tts_c
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.1f} GB")

## 8. Results summary & export

In [ ]:
import json
import statistics

all_results = results_a + results_b + results_c

# Lưu raw results
with open("tts_benchmark_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

# Summary table
print("=" * 70)
print(f"{'Config':<20} {'TTFA median':>12} {'TTFA p95':>10} {'Total median':>13} {'Total p95':>10}")
print("-" * 70)

for name, results in [("A_GGUF_INT4", results_a), ("B_PyTorch_fp32", results_b), ("C_LMDeploy_bf16", results_c)]:
    ttfas = sorted([r["ttfa_ms"] for r in results if r["ttfa_ms"]])
    totals = sorted([r["total_ms"] for r in results])
    
    def p95(lst): return lst[int(len(lst) * 0.95)] if lst else 0
    def median(lst): return statistics.median(lst) if lst else 0
    
    print(f"{name:<20} {median(ttfas):>10.0f}ms {p95(ttfas):>8.0f}ms {median(totals):>11.0f}ms {p95(totals):>8.0f}ms")

print("=" * 70)
print(f"\nAudio files saved to: {OUTPUT_DIR}/")
print("Format: <config>_q<nn>.wav — 10 files per config = 30 files total")
print("Dùng các file WAV này cho CMOS evaluation.")

## 9. Listen & inspect audio samples

Chạy cell này để nghe thử audio từ mỗi config với cùng 1 query.

In [ ]:
from IPython.display import Audio, display
import soundfile as sf

# Nghe query số 1 từ cả 3 config để so sánh trực tiếp
for config_prefix in ["A_GGUF_INT4", "B_PyTorch_fp32", "C_LMDeploy_bf16"]:
    wav_path = OUTPUT_DIR / f"{config_prefix}_q01.wav"
    if wav_path.exists():
        audio, sr = sf.read(str(wav_path))
        print(f"\n--- {config_prefix} (query 1: '{TEST_QUERIES[0]}') ---")
        display(Audio(audio, rate=sr))
    else:
        print(f"{config_prefix}_q01.wav not found")

## 10. Cleanup vLLM server

In [ ]:
vllm_proc.terminate()
vllm_proc.wait()
print("vLLM server stopped.")

gc.collect()
torch.cuda.empty_cache()
print(f"Final VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.1f} GB")